# Learning to Learn: MAML, Knowledge Distillation as 'Learn to Teach', and EWC against Catastrophic Forgetting

**Meta-learning is "learning to learn."** Ordinary machine learning fixes a learning algorithm and uses it to learn the *parameters* of a single task. Meta-learning goes one level up: it learns the **learning procedure itself** (or a component of it) across a whole *distribution of tasks* — treating tasks the way ordinary learning treats individual examples.

This notebook is built on a single spine — the same **"learn to X"** idea, made concrete three ways with small, fully runnable demos:

1. **Learn to INITIALIZE (MAML)** — on few-shot sine-wave regression, a meta-learned initialization adapts to a brand-new task in a handful of gradient steps, far faster than a random or a multi-task-pretrained init.
2. **Learn to TEACH (Knowledge Distillation)** — a student matches a teacher's temperature-softened "soft labels"; then we meta-optimize the *teaching signal* (temperature) for the student's resulting validation loss.
3. **Learn to NOT FORGET (EWC)** — a network trained sequentially on two tasks catastrophically forgets the first; EWC anchors important weights to mitigate it, motivating *learned* anti-forgetting.

**Demo philosophy:** everything is toy / MNIST-scale, CPU-friendly, and runs end-to-end in one class session with **Run All**.

## Learning Objectives & Roadmap

- **LO1 — The meta-learning frame.** Meta-learning learns a *component of the learning algorithm* across a task distribution. *(C01–C03)*
- **LO2 — MAML / Learn to Initialize.** Support/query split, inner-loop adaptation from a shared init φ, outer-loop meta-update of φ; contrast with ordinary pretraining/transfer. *(C05, C12)*
- **LO3 — MAML in action.** A meta-learned init adapts a new sine task in a few steps, beating random and pretrained inits — interactively and quantitatively. *(C06–C11)*
- **LO4 — Knowledge Distillation & Learn to Teach.** Temperature-scaled soft labels transfer 'dark knowledge'; the teaching temperature is itself optimizable. *(C13–C18)*
- **LO5 — Catastrophic Forgetting & EWC.** Sequential training forgets; EWC's Fisher-weighted anchor trades stability vs plasticity; meta-learning as learned anti-forgetting. *(C19–C24)*

**Roadmap:** meta-learning core (C03) → setup (C04) → **Demo 1 / MAML** framing (C05), task distribution (C06), model + functional MAML (C07), meta-training (C08), baseline inits (C09), interactive explorer (C10), adaptation curves (C11), interpretation (C12) → **Demo 2 / KD** framing (C13), data + models (C14), train teacher (C15), temperature explorer (C16), distill vs scratch (C17), learn-to-teach search (C18) → **Demo 3 / EWC** framing (C19), continual setup (C20), forgetting baseline (C21), EWC (C22), strength explorer (C23), interpretation (C24) → synthesis (C25).

All three demos are the **same** meta-learning idea applied to a different learned object: the **initialization**, the **teaching signal**, and the **regularizer**.

## The Meta-Learning Core (shared by all three demos)

Every demo below shares one structure.

**Task distribution p(T).** Instead of one dataset, we have a *distribution of tasks*. Each task T has its own small **support** set (its 'training' examples) and **query** set (its 'test' examples).

| Ordinary learning | Meta-learning |
|---|---|
| One task | A distribution of tasks p(T) |
| Learn the **parameters** | Learn a **component of the algorithm** (init / teaching signal / regularizer) |
| Examples are the unit of data | **Tasks** are the unit of data |

**Two nested loops.**
- **Inner loop** — *adapt within a task*: given a task's support set, take a few gradient steps to fit it.
- **Outer loop** — *update the meta-knowledge across tasks*: improve the shared component (e.g. the initialization φ) so that inner-loop adaptation works well on the **query** set, averaged over many tasks.

**Vocabulary reused throughout:** *meta-train* vs *meta-test* tasks, *support* vs *query*, *inner loop* vs *outer loop*.

**A recurring meta-insight (previewed here, revisited at the end):** because meta-learning treats *tasks like examples*, it inherits ordinary-ML pathologies **one level up** — task-distribution shift (a generalization / domain-adaptation problem over tasks) and forgetting over a *stream* of tasks (meta-level catastrophic forgetting).

In [ ]:
# C04 - Environment setup (single dependency-providing cell)
# numpy/torch/torchvision/matplotlib are preinstalled on Colab; only ipywidgets may be missing.
WIDGETS_OK = True
try:
    import ipywidgets as widgets
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
    try:
        import ipywidgets as widgets
    except Exception:
        WIDGETS_OK = False
        widgets = None
        print('WARNING: ipywidgets unavailable - interactive cells fall back to static draws.')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import copy, math, random
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed: int = 0) -> None:
    # Seed python-random, numpy and torch (+cuda) for reproducible Run All.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)  # make the import cell itself deterministic

try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True

set_seed(0); _a = torch.randn(3)
set_seed(0); _b = torch.randn(3)
assert torch.allclose(_a, _b), 'set_seed is not deterministic'

_w = 'on' if WIDGETS_OK else 'OFF'
print(f'torch {torch.__version__} on {DEVICE} | widgets={_w} | CPU-friendly (toy / small-MNIST scale).')

## Demo 1 — Learn to Initialize: MAML

**The few-shot task.** Each task T regresses a sine wave

$$y = a\,\sin(x + b),$$

with a task-specific amplitude $a$ and phase $b$. You see only $K$ **support** points and must predict the **query** points. With so few points and an unknown $(a, b)$, the *initialization you start adapting from* matters enormously.

**MAML (Model-Agnostic Meta-Learning).** Learn a single initialization $\phi$ such that, after a few gradient steps on a task's support set, the model fits that task well.

- **Inner loop (adapt one task):** $\;\theta' = \phi - \alpha\,\nabla_{\phi}\,\mathcal{L}_{\text{support}}(\phi)$
- **Outer loop (meta-update φ):** minimize the *post-adaptation* query loss across many tasks: $\;\min_{\phi}\,\sum_{T}\,\mathcal{L}^{T}_{\text{query}}(\theta'_{T}).$

The outer loop differentiates **through** the inner adaptation, so $\phi$ is optimized *for adaptability*, not for any single task.

**Contrast with ordinary pretraining / transfer (the SSL route).** Training one model jointly on all tasks also produces an initialization — but it is optimized to fit the *average* task, **not** to adapt fast. This is the **'learning gap'** between a pretext / average objective and the downstream few-shot objective.

**Prediction:** the meta-init will snap onto a new task in **1–5 steps**; the pretrained init drifts slowly; the random init barely moves.

In [ ]:
# C06 - The sine-wave task distribution
K_SHOT, Q_QUERY = 5, 50
X_RANGE     = (-5.0, 5.0)
AMP_RANGE   = (0.1, 5.0)
PHASE_RANGE = (0.0, math.pi)

def sample_sine_task(seed=None, k_shot=None) -> dict:
    # If seed is given, use a LOCAL RandomState so an explicit seed (e.g. the widget
    # task-seed slider) never perturbs the global RNG stream used by Run All.
    rng = np.random.RandomState(seed) if seed is not None else np.random
    k = int(k_shot) if k_shot is not None else K_SHOT
    assert k >= 1, 'k_shot must be >= 1'
    a = rng.uniform(*AMP_RANGE)
    b = rng.uniform(*PHASE_RANGE)
    def _pts(n):
        x = rng.uniform(X_RANGE[0], X_RANGE[1], size=(n, 1)).astype(np.float32)
        y = (a * np.sin(x + b)).astype(np.float32)
        return torch.from_numpy(x).to(DEVICE), torch.from_numpy(y).to(DEVICE)
    sx, sy = _pts(k)
    qx, qy = _pts(Q_QUERY)
    return {'support_x': sx, 'support_y': sy, 'query_x': qx, 'query_y': qy,
            'a': float(a), 'b': float(b)}

def sample_task_batch(n_tasks: int) -> list:
    # A batch of distinct tasks drawn from the global RNG (seed=None).
    return [sample_sine_task(seed=None) for _ in range(n_tasks)]

set_seed(1)
dense = torch.linspace(X_RANGE[0], X_RANGE[1], 200).reshape(-1, 1)
plt.figure()
for i in range(5):
    t = sample_sine_task(seed=10 + i)
    yt = t['a'] * np.sin(dense.numpy() + t['b'])
    line, = plt.plot(dense.numpy(), yt, lw=1.5, label=f"a={t['a']:.1f}, b={t['b']:.1f}")
    plt.scatter(t['support_x'].cpu(), t['support_y'].cpu(), color=line.get_color(), s=30, zorder=3)
plt.title('Sampled sine-regression tasks (true curve + K support points)')
plt.xlabel('x'); plt.ylabel('y'); plt.legend(fontsize=7); plt.show()
print(f'K_SHOT={K_SHOT} support points, Q_QUERY={Q_QUERY} query points per task.')

In [ ]:
# C07 - MLP regressor as functional params + differentiable inner-loop adaptation
HIDDEN = 40  # fixed width for every init, so comparisons isolate phi (not capacity)

def make_mlp(seed: int) -> list:
    # Ordered leaf params [W1,b1,W2,b2,W3,b3] of a 1->HIDDEN->HIDDEN->1 MLP. This IS phi.
    set_seed(seed)
    def w(out_f, in_f):
        t = torch.empty(out_f, in_f)
        nn.init.kaiming_uniform_(t, a=math.sqrt(5))
        return t.to(DEVICE).requires_grad_(True)
    def bvec(out_f):
        return torch.zeros(out_f, device=DEVICE, requires_grad=True)
    return [w(HIDDEN, 1), bvec(HIDDEN), w(HIDDEN, HIDDEN), bvec(HIDDEN), w(1, HIDDEN), bvec(1)]

def functional_forward(params: list, x: torch.Tensor) -> torch.Tensor:
    W1, b1, W2, b2, W3, b3 = params
    h = F.relu(F.linear(x, W1, b1))
    h = F.relu(F.linear(h, W2, b2))
    return F.linear(h, W3, b3)

def inner_adapt(params, support_x, support_y, steps, lr, second_order) -> list:
    # `steps` SGD steps on the support MSE from params. Returns a NEW list (input untouched).
    # create_graph=second_order -> true 2nd-order MAML when True, cheap 1st-order when False.
    adapted = list(params)
    for _ in range(steps):
        loss = F.mse_loss(functional_forward(adapted, support_x), support_y)
        grads = torch.autograd.grad(loss, adapted, create_graph=second_order)
        adapted = [p - lr * g for p, g in zip(adapted, grads)]
    return adapted

_p = make_mlp(0)
assert len(_p) == 6 and all(t.requires_grad and t.device == DEVICE for t in _p)
assert functional_forward(_p, torch.zeros(3, 1, device=DEVICE)).shape == (3, 1)
_ = inner_adapt(_p, torch.randn(5, 1, device=DEVICE), torch.randn(5, 1, device=DEVICE), 2, 0.01, False)
assert all(torch.equal(x, y) for x, y in zip(_p, make_mlp(0))), 'inner_adapt mutated phi!'
print(f'MLP 1 -> {HIDDEN} -> {HIDDEN} -> 1 ({sum(t.numel() for t in _p)} params); inner_adapt ready.')

In [ ]:
# C08 - Meta-train MAML (learn the initialization phi)
set_seed(0)
phi_maml = make_mlp(seed=1)
meta_opt = torch.optim.Adam(phi_maml, lr=1e-3)

META_ITERS, TASK_BATCH = 2000, 8
INNER_STEPS, INNER_LR  = 1, 0.01
SECOND_ORDER = True   # cheap at this scale; set False for first-order MAML if slow

maml_history = []
for it in range(META_ITERS):
    tasks = sample_task_batch(TASK_BATCH)
    meta_loss = 0.0
    for t in tasks:
        adapted = inner_adapt(phi_maml, t['support_x'], t['support_y'], INNER_STEPS, INNER_LR, SECOND_ORDER)
        meta_loss = meta_loss + F.mse_loss(functional_forward(adapted, t['query_x']), t['query_y'])
    meta_loss = meta_loss / TASK_BATCH
    meta_opt.zero_grad()
    meta_loss.backward()
    meta_opt.step()
    maml_history.append(meta_loss.item())
    if (it + 1) % 400 == 0:
        print(f'  meta-iter {it+1:4d}/{META_ITERS}  meta-loss {meta_loss.item():.4f}')

phi_random = make_mlp(seed=999)  # untrained baseline init (never optimized)

plt.figure()
plt.plot(maml_history, lw=1)
plt.yscale('log'); plt.xlabel('meta-iteration'); plt.ylabel('meta-loss (query MSE)')
plt.title('MAML meta-training loss'); plt.show()

_t = sample_sine_task(seed=12345)
def _q_mse(phi):
    a = inner_adapt(phi, _t['support_x'], _t['support_y'], INNER_STEPS, INNER_LR, False)
    return F.mse_loss(functional_forward(a, _t['query_x']), _t['query_y']).item()
print(f'held-out query MSE after {INNER_STEPS} step(s): phi_maml={_q_mse(phi_maml):.3f} vs phi_random={_q_mse(phi_random):.3f}')
assert len(maml_history) == META_ITERS and maml_history[-1] < maml_history[0]

In [ ]:
# C09 - Multi-task 'pretraining' init (the transfer/SSL stand-in): fit the pooled 'average'
# task with NO inner-loop adaptation, so it is reasonable but not optimized for fast adaptation.
set_seed(2)
phi_pretrain = make_mlp(seed=2)
opt = torch.optim.Adam(phi_pretrain, lr=1e-3)
PRETRAIN_ITERS, POOL_TASKS = 2000, 8

pretrain_history = []
for it in range(PRETRAIN_ITERS):
    tasks = sample_task_batch(POOL_TASKS)
    X = torch.cat([t['support_x'] for t in tasks] + [t['query_x'] for t in tasks], dim=0)
    Y = torch.cat([t['support_y'] for t in tasks] + [t['query_y'] for t in tasks], dim=0)
    loss = F.mse_loss(functional_forward(phi_pretrain, X), Y)  # pooled fit, no adaptation
    opt.zero_grad(); loss.backward(); opt.step()
    pretrain_history.append(loss.item())
    if (it + 1) % 400 == 0:
        print(f'  pretrain-iter {it+1:4d}/{PRETRAIN_ITERS}  pooled MSE {loss.item():.4f}')

plt.figure()
plt.plot(pretrain_history, lw=1, color='tab:orange')
plt.yscale('log'); plt.xlabel('iteration'); plt.ylabel('pooled MSE')
plt.title('Multi-task pretraining loss (fits the average task)'); plt.show()

assert all(p.shape == q.shape for p, q in zip(phi_pretrain, make_mlp(0)))
print('phi_pretrain ready - a learned-but-not-for-adaptation init (varying a,b => pooled target ~ mean sine, so it adapts slowly).')

In [ ]:
# C10 - Interactive MAML adaptation explorer (centerpiece widget for Demo 1)
_dense_x = torch.linspace(X_RANGE[0], X_RANGE[1], 200).reshape(-1, 1).to(DEVICE)
_INITS = [('MAML', phi_maml, 'tab:green'),
          ('Pretrain', phi_pretrain, 'tab:orange'),
          ('Random', phi_random, 'tab:red')]

def draw_adaptation(inner_steps, k_shots, task_seed):
    task = sample_sine_task(seed=task_seed, k_shot=k_shots)
    xs_np = _dense_x.cpu().numpy()
    true_y = task['a'] * np.sin(xs_np + task['b'])
    plt.figure()
    plt.plot(xs_np, true_y, 'k--', lw=2, label='true sine')
    plt.scatter(task['support_x'].cpu(), task['support_y'].cpu(), color='black', s=45, zorder=5, label=f'{k_shots} support pts')
    bits = []
    for name, phi, color in _INITS:
        adapted = inner_adapt(phi, task['support_x'], task['support_y'], inner_steps, INNER_LR, False)
        pred = functional_forward(adapted, _dense_x).detach().cpu().numpy()
        q_mse = F.mse_loss(functional_forward(adapted, task['query_x']), task['query_y']).item()
        plt.plot(xs_np, pred, color=color, lw=1.8, label=name)
        bits.append(f'{name} {q_mse:.2f}')
    plt.title(f'steps={inner_steps}, K={k_shots}, seed={task_seed} | query MSE: ' + ', '.join(bits))
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(fontsize=8); plt.ylim(-6, 6); plt.show()

maml_adaptation_explorer = None
if WIDGETS_OK:
    try:
        s_steps = widgets.IntSlider(value=5, min=0, max=10, description='inner steps')
        s_k     = widgets.IntSlider(value=5, min=1, max=10, description='K shots')
        s_seed  = widgets.IntSlider(value=0, min=0, max=50, description='task seed')
        out = widgets.interactive_output(draw_adaptation, {'inner_steps': s_steps, 'k_shots': s_k, 'task_seed': s_seed})
        maml_adaptation_explorer = widgets.VBox([widgets.HBox([s_steps, s_k, s_seed]), out])
        display(maml_adaptation_explorer)
    except Exception as e:
        print(f'Widget init failed ({e}); drawing static default.')
        draw_adaptation(5, 5, 0)
else:
    draw_adaptation(5, 5, 0)

In [ ]:
# C11 - Quantify the adaptation advantage (mean query MSE vs inner steps)
N_EVAL_TASKS, MAX_STEPS, EVAL_LR = 100, 5, INNER_LR
set_seed(7)

def evaluate_init(phi, task, max_steps, lr):
    # Per-step query MSE for one task (length max_steps+1).
    out = []
    for s in range(max_steps + 1):
        adapted = inner_adapt(phi, task['support_x'], task['support_y'], s, lr, False)
        out.append(F.mse_loss(functional_forward(adapted, task['query_x']), task['query_y']).item())
    return out

names = ['MAML', 'Pretrain', 'Random']
phis  = {'MAML': phi_maml, 'Pretrain': phi_pretrain, 'Random': phi_random}
colors = {'MAML': 'tab:green', 'Pretrain': 'tab:orange', 'Random': 'tab:red'}
curves = {n: np.zeros((N_EVAL_TASKS, MAX_STEPS + 1)) for n in names}
for i in range(N_EVAL_TASKS):
    task = sample_sine_task(seed=1000 + i)   # one task, all three inits -> paired comparison
    for n in names:
        curves[n][i] = evaluate_init(phis[n], task, MAX_STEPS, EVAL_LR)

maml_adapt_curves = plt.figure()
xs = np.arange(MAX_STEPS + 1)
maml_results = {}
for n in names:
    mean, std = curves[n].mean(0), curves[n].std(0)
    plt.plot(xs, mean, '-o', color=colors[n], label=n)
    plt.fill_between(xs, mean - std, mean + std, color=colors[n], alpha=0.15)
    maml_results[n] = {'mse_0': float(mean[0]), 'mse_1': float(mean[1]),
                       f'mse_{MAX_STEPS}': float(mean[MAX_STEPS]),
                       'mean_curve': [float(v) for v in mean]}
plt.yscale('log'); plt.xlabel('inner adaptation steps'); plt.ylabel('mean query MSE')
plt.title(f'Adaptation curves over {N_EVAL_TASKS} held-out tasks'); plt.legend(); plt.show()

hdr = ['init', '0 steps', '1 step', f'{MAX_STEPS} steps']
print(f'{hdr[0]:10s}{hdr[1]:>10s}{hdr[2]:>10s}{hdr[3]:>12s}')
for n in names:
    r = maml_results[n]
    m0, m1, mN = r['mse_0'], r['mse_1'], r[f'mse_{MAX_STEPS}']
    print(f'{n:10s}{m0:10.3f}{m1:10.3f}{mN:12.3f}')
assert maml_results['MAML'][f'mse_{MAX_STEPS}'] < maml_results['Random'][f'mse_{MAX_STEPS}']

## Interpreting Demo 1 — why the meta-init wins

The adaptation curves make the **'learn to initialize'** idea concrete:

- **MAML init** — φ was *explicitly optimized* so that a few gradient steps minimize the query loss. It sits in a region of weight space from which **every task is a short walk away**, so 1–5 steps suffice.
- **Pretrained init** — learned good *average* features, but was never optimized for adaptability. This is the **'learning gap'**: a pretext / average objective ≠ the downstream few-shot objective. It may start lower than random at 0 steps yet still adapt slowly.
- **Random init** — neither good features nor adaptability.

**Takeaway:** both *learned* inits beat random, but only MAML's was learned **FOR learning**. This is exactly the **meta-learning vs. self-supervised / transfer** contrast: pretraining gives you a good starting point; meta-learning gives you a starting point *that is good at moving*.

**Bridge to Demo 2:** the learned object need not be the initialization. Next we keep the model fixed and instead learn the **teaching signal** — *Learn to Teach*.

## Demo 2 — Learn to Teach: Knowledge Distillation

**Vanilla knowledge distillation (KD).** Train a large **teacher** on labeled data. Then train a small **student** to match the teacher's *temperature-softened* output distribution — the **soft labels** — via a KL / cross-entropy term, optionally mixed with the ordinary hard-label loss:

$$\mathcal{L}_{\text{KD}} = \alpha\,T^{2}\,\mathrm{KL}\!\big(\mathrm{softmax}(z_s/T)\,\|\,\mathrm{softmax}(z_t/T)\big) + (1-\alpha)\,\mathrm{CE}(z_s, y).$$

**Why temperature?** The softmax temperature $T$ *flattens* the distribution, exposing **'dark knowledge'** — the teacher's relative similarities among the *wrong* classes (a handwritten **3** looks more like an **8** than a **1**). A hard one-hot label throws that structure away.

**The meta-learning insight — 'Learn to Teach'.** The teacher was trained to **classify**, not to **teach**, so its signal is *not optimal* for the student. *Learn to Teach* (e.g. **BERT Learns to Teach**, ACL 2022 / Meta Knowledge Distillation) **meta-optimizes the teaching signal to minimize the student's resulting test loss** — the teacher's signal becomes a learned object, just like MAML's initialization.

**Runnable plan:** train a teacher (C15) → explore temperature interactively (C16) → distill a student vs. a from-scratch student (C17) → **meta-search the teaching temperature** for the student's validation accuracy (C18).

In [ ]:
# C14 - Small MNIST subset + teacher / student architectures
set_seed(0)
DIGIT_CLASSES = list(range(10))
N_TRAIN, N_TEST = 4000, 1000

def load_mnist_subset(n_train, n_test):
    # Balanced flat MNIST subset on DEVICE; falls back to synthetic blobs if offline.
    try:
        tr = torchvision.datasets.MNIST('./data', train=True, download=True)
        te = torchvision.datasets.MNIST('./data', train=False, download=True)
        def pack(ds, n):
            X = ds.data.float().div(255).sub(0.1307).div(0.3081).reshape(len(ds), -1)
            idx = torch.randperm(len(ds))[:n]
            return X[idx].to(DEVICE), ds.targets[idx].to(DEVICE)
        xtr, ytr = pack(tr, n_train)
        xte, yte = pack(te, n_test)
        print(f'Loaded MNIST subset: {n_train} train / {n_test} test.')
    except Exception as e:
        print(f'WARNING: MNIST download failed ({e}); using synthetic 784-dim blobs.')
        g = torch.Generator().manual_seed(0)
        centers = torch.randn(10, 784, generator=g) * 3.0
        def blobs(n):
            y = torch.randint(0, 10, (n,), generator=g)
            X = centers[y] + torch.randn(n, 784, generator=g)
            return X.to(DEVICE), y.to(DEVICE)
        xtr, ytr = blobs(n_train)
        xte, yte = blobs(n_test)
    return {'x': xtr, 'y': ytr}, {'x': xte, 'y': yte}

mnist_train, mnist_test = load_mnist_subset(N_TRAIN, N_TEST)

def make_teacher():
    return nn.Sequential(nn.Linear(784, 256), nn.ReLU(),
                         nn.Linear(256, 256), nn.ReLU(),
                         nn.Linear(256, 10)).to(DEVICE)

def make_student():
    return nn.Sequential(nn.Linear(784, 32), nn.ReLU(),
                         nn.Linear(32, 10)).to(DEVICE)

assert mnist_train['x'].shape[1] == 784 and mnist_train['y'].dtype == torch.long
assert sum(p.numel() for p in make_teacher().parameters()) > sum(p.numel() for p in make_student().parameters())

fig, axes = plt.subplots(3, 6, figsize=(8, 4))
for ax, idx in zip(axes.ravel(), range(18)):
    ax.imshow(mnist_train['x'][idx].cpu().reshape(28, 28), cmap='gray')
    ax.set_title(str(mnist_train['y'][idx].item()), fontsize=9); ax.axis('off')
fig.suptitle('Sample MNIST digits'); plt.tight_layout(); plt.show()

In [ ]:
# C15 - Train the teacher + illustrate the temperature effect
set_seed(0)

def accuracy(model, data):
    model.eval()
    with torch.no_grad():
        return (model(data['x']).argmax(1) == data['y']).float().mean().item()

def train_classifier(model, data, epochs, lr, batch=128):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = data['x'].shape[0]; hist = []
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEVICE)
        model.train()
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            loss = F.cross_entropy(model(data['x'][idx]), data['y'][idx])
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append(loss.item())
    return hist

TEACHER_EPOCHS, BATCH = 5, 128
teacher = make_teacher()
train_classifier(teacher, mnist_train, TEACHER_EPOCHS, lr=1e-3, batch=BATCH)
teacher.eval()
teacher_acc = accuracy(teacher, mnist_test)
print(f'teacher test accuracy = {teacher_acc:.4f}')

idx = 0
with torch.no_grad():
    logits = teacher(mnist_test['x'][idx:idx + 1]).squeeze()
p1 = F.softmax(logits / 1.0, dim=0).cpu().numpy()
p4 = F.softmax(logits / 4.0, dim=0).cpu().numpy()
true_label = mnist_test['y'][idx].item()
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for a, p, ttl in [(ax[0], p1, 'T=1 (peaky)'), (ax[1], p4, 'T=4 (dark knowledge)')]:
    bars = a.bar(DIGIT_CLASSES, p, color='tab:blue')
    bars[true_label].set_color('tab:green')
    a.set_title(ttl); a.set_xticks(DIGIT_CLASSES); a.set_xlabel('class'); a.set_ylim(0, 1)
fig.suptitle(f'Teacher soft labels for a {true_label} (green = true class)')
plt.tight_layout(); plt.show()
assert not teacher.training

In [ ]:
# C16 - Interactive temperature / soft-label explorer (Demo 2 widget)
def _pick_confusable():
    # Pick a test digit whose top-2 teacher probs are close, so dark knowledge is visible.
    teacher.eval()
    with torch.no_grad():
        probs = F.softmax(teacher(mnist_test['x'][:200]), dim=1)
    top2 = probs.topk(2, dim=1).values
    return int((top2[:, 0] - top2[:, 1]).argmin().item())

default_idx = _pick_confusable()

def draw_softlabels(temperature, img_index):
    temperature = max(float(temperature), 0.5)  # guard divide-by-zero
    x = mnist_test['x'][img_index:img_index + 1]
    teacher.eval()
    with torch.no_grad():
        probs = F.softmax(teacher(x) / temperature, dim=1).squeeze().cpu().numpy()
    true_label = mnist_test['y'][img_index].item()
    pred = int(probs.argmax())
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
    ax[0].imshow(x.cpu().reshape(28, 28), cmap='gray')
    ax[0].set_title(f'digit (true={true_label})'); ax[0].axis('off')
    bars = ax[1].bar(DIGIT_CLASSES, probs, color='tab:blue')
    bars[true_label].set_color('tab:green')
    ax[1].set_title(f'T={temperature:.1f}, pred={pred}')
    ax[1].set_xticks(DIGIT_CLASSES); ax[1].set_ylim(0, 1); ax[1].set_xlabel('class')
    plt.tight_layout(); plt.show()

temperature_explorer = None
if WIDGETS_OK:
    try:
        s_T = widgets.FloatSlider(value=4.0, min=0.5, max=10.0, step=0.5, description='T')
        s_i = widgets.IntSlider(value=default_idx, min=0, max=len(mnist_test['y']) - 1, description='img idx')
        out = widgets.interactive_output(draw_softlabels, {'temperature': s_T, 'img_index': s_i})
        temperature_explorer = widgets.VBox([widgets.HBox([s_T, s_i]), out])
        display(temperature_explorer)
    except Exception as e:
        print(f'Widget init failed ({e}); drawing static default.')
        draw_softlabels(4.0, default_idx)
else:
    draw_softlabels(4.0, default_idx)

In [ ]:
# C17 - Distill the student vs. train an identical student from scratch
set_seed(0)

def kd_loss(student_logits, teacher_logits, hard_labels, T, alpha):
    soft = F.kl_div(F.log_softmax(student_logits / T, dim=1),
                    F.softmax(teacher_logits / T, dim=1),
                    reduction='batchmean') * (T * T)
    hard = F.cross_entropy(student_logits, hard_labels)
    return alpha * soft + (1 - alpha) * hard

KD_T, KD_ALPHA, STUDENT_EPOCHS, BATCH = 4.0, 0.7, 8, 128

def distill_student(T, epochs, alpha, seed, data=None):
    # Fresh student trained with the KD loss; returns (model, per-epoch test-acc history).
    data = data if data is not None else mnist_train
    set_seed(seed)
    student = make_student()
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    n = data['x'].shape[0]; hist = []
    teacher.eval()
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEVICE)
        student.train()
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            with torch.no_grad():
                t_logits = teacher(data['x'][idx])
            s_logits = student(data['x'][idx])
            loss = kd_loss(s_logits, t_logits, data['y'][idx], T, alpha)
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append(accuracy(student, mnist_test))
    return student, hist

def train_scratch(epochs, seed):
    set_seed(seed)
    student = make_student()
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    n = mnist_train['x'].shape[0]; hist = []
    for ep in range(epochs):
        perm = torch.randperm(n, device=DEVICE)
        student.train()
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            loss = F.cross_entropy(student(mnist_train['x'][idx]), mnist_train['y'][idx])
            opt.zero_grad(); loss.backward(); opt.step()
        hist.append(accuracy(student, mnist_test))
    return student, hist

student_kd, kd_hist = distill_student(KD_T, STUDENT_EPOCHS, KD_ALPHA, seed=123)
student_scratch, scratch_hist = train_scratch(STUDENT_EPOCHS, seed=123)
kd_history = {'kd': kd_hist, 'scratch': scratch_hist}
kd_acc = accuracy(student_kd, mnist_test)
scratch_acc = accuracy(student_scratch, mnist_test)

ep_axis = range(1, STUDENT_EPOCHS + 1)
plt.figure()
plt.plot(ep_axis, kd_hist, '-o', label=f'distilled (T={KD_T})')
plt.plot(ep_axis, scratch_hist, '-o', label='from scratch')
plt.xlabel('epoch'); plt.ylabel('test accuracy')
plt.title('Distilled student vs identical-capacity scratch student'); plt.legend(); plt.show()
print(f'kd_acc = {kd_acc:.4f}   scratch_acc = {scratch_acc:.4f}   (teacher = {teacher_acc:.4f})')

In [ ]:
# C18 - Learn to Teach: one-parameter meta-search over teaching temperature.
# Simplified stand-in for full bi-level learn-to-teach (which would differentiate the
# student's test loss through training to update the WHOLE teacher). Here the single
# teaching parameter is the distillation temperature T; the meta-objective is the
# student's VALIDATION accuracy on a split carved from mnist_train (NOT mnist_test).
set_seed(0)
TEMP_GRID = [1, 2, 3, 4, 6, 8, 10]
QUICK_EPOCHS = 4

n = mnist_train['x'].shape[0]
perm = torch.randperm(n, device=DEVICE)
val_idx, tr_idx = perm[:800], perm[800:]
val_split   = {'x': mnist_train['x'][val_idx], 'y': mnist_train['y'][val_idx]}
train_split = {'x': mnist_train['x'][tr_idx], 'y': mnist_train['y'][tr_idx]}

def search_teaching_temperature(temp_grid, epochs, val_data):
    curve = []
    for T in temp_grid:
        student, _ = distill_student(T=float(T), epochs=epochs, alpha=KD_ALPHA, seed=123, data=train_split)
        acc = accuracy(student, val_data)
        curve.append((float(T), acc))
        print(f'  T={T:5.1f} -> student val acc {acc:.4f}')
    return curve

teach_curve = search_teaching_temperature(TEMP_GRID, QUICK_EPOCHS, val_split)
best_T = max(teach_curve, key=lambda p: p[1])[0]

learn_to_teach_fig = plt.figure()
Ts   = [t for t, _ in teach_curve]
accs = [a for _, a in teach_curve]
plt.plot(Ts, accs, '-o', color='tab:purple')
plt.axvline(best_T, color='tab:green', ls='--', label=f'meta best T={best_T:g}')
plt.axvline(KD_T, color='tab:gray', ls=':', label=f'C17 default T={KD_T:g}')
plt.xlabel('teaching temperature T'); plt.ylabel('student validation accuracy')
plt.title('Learn to Teach: student val accuracy vs teaching temperature')
plt.legend(); plt.show()
print(f'meta-selected best_T = {best_T:g}  (vs naive default T = {KD_T:g})')
assert len(teach_curve) == len(TEMP_GRID) and best_T in TEMP_GRID

## Demo 3 — Learn to Not Forget: Catastrophic Forgetting & EWC

**Continual learning.** A single network is trained on **Task A**, then on **Task B**, with Task A's data *no longer available*. Naive SGD on B overwrites the weights that solved A, so accuracy on A **collapses** — **catastrophic forgetting**.

**The regularization-based fix (EWC).** Estimate each parameter's **importance** for Task A — Elastic Weight Consolidation uses the diagonal of the **Fisher information** at the Task-A optimum $\theta^{*}$ — then, while learning B, add a penalty that anchors important weights near their Task-A values:

$$\mathcal{L}_B(\theta) + \frac{\lambda}{2}\sum_i F_i\,(\theta_i - \theta^{*}_i)^2.$$

Important weights (large $F_i$) are held in place; unimportant ones stay free to learn B. Plain L2 (which penalizes *all* weights equally) works far worse, because it cannot tell which weights mattered.

**The meta angle.** EWC's importance measure and penalty are **hand-designed**. Meta-learning can instead **LEARN** a regularizer — or an initialization — that *intrinsically* resists forgetting ('learn to not forget'). And the higher-order twist previewed in C03 returns: a meta-learner trained over a *stream* of tasks can itself **meta-forget** earlier tasks (**Online Meta-Learning**, ICML 2019).

**Prediction:** naive SGD forgets A; EWC at a good $\lambda$ keeps **both**.

In [ ]:
# C20 - Two visualizable 2D continual-learning tasks
set_seed(0)

def make_task_2d(n_per_class, angle, shift=(0, 0), seed=None):
    # Two Gaussian blobs (classes 0/1) in 2D, rotated by `angle` and shifted by `shift`,
    # so Task A and Task B have different decision boundaries. Returns train+test tensors.
    rng = np.random.RandomState(seed)
    base = np.array([[-2.0, 0.0], [2.0, 0.0]], dtype=np.float32)  # class means
    R = np.array([[math.cos(angle), -math.sin(angle)],
                  [math.sin(angle),  math.cos(angle)]], dtype=np.float32)
    means = base @ R.T + np.array(shift, dtype=np.float32)
    def gen(n):
        xs, ys = [], []
        for c in (0, 1):
            xs.append(rng.randn(n, 2).astype(np.float32) * 0.7 + means[c])
            ys.append(np.full(n, c, dtype=np.int64))
        X = torch.from_numpy(np.concatenate(xs)).to(DEVICE)
        Y = torch.from_numpy(np.concatenate(ys)).to(DEVICE)
        return X, Y
    x, y = gen(n_per_class)
    xt, yt = gen(max(n_per_class // 2, 1))
    return {'x': x, 'y': y, 'x_test': xt, 'y_test': yt}

task_A = make_task_2d(100, angle=0.0, seed=1)
task_B = make_task_2d(100, angle=math.pi / 2, seed=2)

def make_classifier(seed):
    set_seed(seed)
    return nn.Sequential(nn.Linear(2, 32), nn.ReLU(),
                         nn.Linear(32, 32), nn.ReLU(),
                         nn.Linear(32, 2)).to(DEVICE)

def train_on_task(model, task, epochs, optimizer, extra_loss=None):
    # Generic full-batch CE loop; extra_loss(model)->scalar plugs in the EWC penalty later.
    hist = []
    model.train()
    for _ in range(epochs):
        loss = F.cross_entropy(model(task['x']), task['y'])
        if extra_loss is not None:
            loss = loss + extra_loss(model)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        hist.append(loss.item())
    return hist

def eval_acc(model, task):
    model.eval()
    with torch.no_grad():
        return (model(task['x_test']).argmax(1) == task['y_test']).float().mean().item()

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
for a, task, ttl in [(ax[0], task_A, 'Task A (angle=0)'), (ax[1], task_B, 'Task B (angle=pi/2)')]:
    X = task['x'].cpu().numpy(); Y = task['y'].cpu().numpy()
    a.scatter(X[Y == 0, 0], X[Y == 0, 1], s=12, color='tab:blue', label='class 0')
    a.scatter(X[Y == 1, 0], X[Y == 1, 1], s=12, color='tab:orange', label='class 1')
    a.set_title(ttl); a.legend(fontsize=8); a.set_xlim(-5, 5); a.set_ylim(-5, 5)
fig.suptitle('Two continual-learning tasks (note the rotated boundary)')
plt.tight_layout(); plt.show()
assert task_A['x'].shape[1] == 2 and task_A['y'].dtype == torch.long

In [ ]:
# C21 - Catastrophic-forgetting baseline (naive sequential training)
set_seed(0)
EPOCHS_A, EPOCHS_B = 200, 100

model = make_classifier(seed=0)
optA = torch.optim.Adam(model.parameters(), lr=1e-2)
train_on_task(model, task_A, EPOCHS_A, optA)
acc_A_initial = eval_acc(model, task_A)

model_after_A = copy.deepcopy(model)  # frozen Task-A optimum (independent snapshot)
theta_star = {n: p.detach().clone() for n, p in model_after_A.named_parameters()}

naive_model = copy.deepcopy(model_after_A)
optB = torch.optim.Adam(naive_model.parameters(), lr=1e-2)  # fresh optimizer for phase B
naive_history = {'A': [], 'B': []}
for ep in range(EPOCHS_B):
    train_on_task(naive_model, task_B, 1, optB)
    naive_history['A'].append(eval_acc(naive_model, task_A))
    naive_history['B'].append(eval_acc(naive_model, task_B))

naive_fig = plt.figure()
plt.plot(naive_history['A'], label='Task A (being forgotten)', color='tab:blue')
plt.plot(naive_history['B'], label='Task B (being learned)', color='tab:orange')
plt.xlabel('Task-B training epoch'); plt.ylabel('accuracy')
plt.title('Catastrophic forgetting: naive sequential training'); plt.legend(); plt.show()
fa, fb = naive_history['A'][-1], naive_history['B'][-1]
print(f'Task-A acc: {acc_A_initial:.3f} (after A) -> {fa:.3f} (after B)  |  Task-B final: {fb:.3f}')
assert acc_A_initial > 0.85

In [ ]:
# C22 - EWC: Fisher-weighted anchoring mitigates forgetting
def compute_fisher(model, task):
    # Diagonal empirical Fisher at the given params: mean over samples of the squared
    # gradient of the log-likelihood at the model's PREDICTED label (empirical Fisher).
    model.eval()
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
    X = task['x']
    for i in range(X.shape[0]):
        model.zero_grad()
        logits = model(X[i:i + 1])
        logp = F.log_softmax(logits, dim=1)
        label = logits.argmax(1).detach()          # predicted label -> empirical Fisher
        F.nll_loss(logp, label).backward()
        for n, p in model.named_parameters():
            if p.grad is not None:
                fisher[n] += p.grad.detach() ** 2
    for n in fisher:
        fisher[n] /= X.shape[0]
    model.zero_grad()
    return fisher

def ewc_penalty(model, theta_star, fisher, lam):
    return lam * sum((fisher[n] * (p - theta_star[n]) ** 2).sum() for n, p in model.named_parameters())

fisher = compute_fisher(model_after_A, task_A)  # computed ONCE at the Task-A optimum

def train_on_B_ewc(lam):
    # Start from a fresh copy of the Task-A optimum; train on B with the EWC penalty.
    ewc_model = copy.deepcopy(model_after_A)
    opt = torch.optim.Adam(ewc_model.parameters(), lr=1e-2)
    hist = {'A': [], 'B': []}
    extra = lambda m: ewc_penalty(m, theta_star, fisher, lam)
    for ep in range(EPOCHS_B):
        train_on_task(ewc_model, task_B, 1, opt, extra_loss=extra)
        hist['A'].append(eval_acc(ewc_model, task_A))
        hist['B'].append(eval_acc(ewc_model, task_B))
    return ewc_model, hist

LAMBDA_DEFAULT = 1e3
ewc_model, ewc_history = train_on_B_ewc(LAMBDA_DEFAULT)

plt.figure()
plt.plot(naive_history['A'], '--', color='tab:blue', label='naive: Task A')
plt.plot(naive_history['B'], '--', color='tab:orange', label='naive: Task B')
plt.plot(ewc_history['A'], '-', color='tab:blue', label='EWC: Task A')
plt.plot(ewc_history['B'], '-', color='tab:orange', label='EWC: Task B')
plt.xlabel('Task-B training epoch'); plt.ylabel('accuracy')
plt.title(f'EWC (lambda={LAMBDA_DEFAULT:g}) retains Task A vs naive forgetting')
plt.legend(fontsize=8); plt.show()
fa_n, fa_e = naive_history['A'][-1], ewc_history['A'][-1]
fb_n, fb_e = naive_history['B'][-1], ewc_history['B'][-1]
print(f'Final Task-A acc:  naive={fa_n:.3f}  EWC={fa_e:.3f}')
print(f'Final Task-B acc:  naive={fb_n:.3f}  EWC={fb_e:.3f}')
assert set(fisher.keys()) == {n for n, _ in model_after_A.named_parameters()}

In [ ]:
# C23 - Interactive EWC-strength explorer (centerpiece widget for Demo 3)
def draw_ewc(lam):
    _, hist = train_on_B_ewc(float(lam))  # reuses cached model_after_A + fisher
    accA, accB = hist['A'][-1], hist['B'][-1]
    naiveA, naiveB = naive_history['A'][-1], naive_history['B'][-1]
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar([0, 1], [accA, accB], color=['tab:blue', 'tab:orange'])
    ax.axhline(naiveA, color='tab:blue', ls=':', alpha=0.6)   # naive (lambda=0) reference
    ax.axhline(naiveB, color='tab:orange', ls=':', alpha=0.6)
    for b, v in zip(bars, [accA, accB]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f'{v:.2f}', ha='center')
    if accA < 0.7:
        regime = 'low lambda -> forgets Task A'
    elif accB < 0.7:
        regime = 'high lambda -> blocks Task B'
    else:
        regime = 'balanced -> keeps both'
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Task A', 'Task B']); ax.set_ylim(0, 1.05)
    ax.set_title(f'lambda={lam:.3g}  ->  {regime}')
    ax.set_xlabel('dotted lines = naive (lambda=0) baseline')
    plt.show()

ewc_strength_explorer = None
if WIDGETS_OK:
    try:
        s_lam = widgets.FloatLogSlider(value=LAMBDA_DEFAULT, base=10, min=-2, max=5, step=0.25, description='lambda')
        out = widgets.interactive_output(draw_ewc, {'lam': s_lam})
        ewc_strength_explorer = widgets.VBox([s_lam, out])
        display(ewc_strength_explorer)
    except Exception as e:
        print(f'Widget init failed ({e}); drawing static default.')
        draw_ewc(LAMBDA_DEFAULT)
else:
    draw_ewc(LAMBDA_DEFAULT)

## Interpreting Demo 3 — the stability–plasticity trade-off

The λ slider exposes the core tension of continual learning directly:

- **λ too small** → the anchor is weak, the network freely overwrites Task-A weights → **forgets A** (the naive regime).
- **λ too large** → important weights are frozen so hard the network **cannot learn B** → plasticity is blocked.
- **Intermediate λ** → important weights stay put while the rest adapt → **both tasks retained**.

EWC's Fisher-weighted anchor is a **hand-designed** notion of parameter importance — a fixed, human-specified rule for *what to protect and how hard*.

**The meta-learning view.** Rather than hand-design the regularizer, **learn it** — or learn an initialization that is *intrinsically* forgetting-resistant — by optimizing across a *stream* of tasks: **'learn to not forget'**.

**Closing the higher-order loop (previewed in C03).** Because meta-learning treats *tasks like examples*, a meta-learner trained on a task **stream** can itself catastrophically forget earlier tasks (**Online Meta-Learning**, ICML 2019). Continual learning reappears **one level up** — the same pathology, meta-shifted.

## Synthesis & Outlook

**One spine, three learned objects.** Every demo was the *same* meta-learning idea — *learn to X* — applied to a different component of the learning procedure, each treating **tasks the way ordinary learning treats examples**:

| Demo | Learned object | Result |
|---|---|---|
| **Learn to INITIALIZE** (MAML, C05–C12) | the initialization φ | adapts a new sine task in a few steps, beating random & pretrained inits |
| **Learn to TEACH** (KD, C13–C18) | the teaching signal (temperature) | distillation transfers dark knowledge; the teaching T is itself optimizable |
| **Learn to NOT FORGET** (EWC, C19–C24) | the regularizer | Fisher-anchoring retains Task A, motivating *learned* anti-forgetting |

**The recurring meta-insight.** Meta-learning inherits ordinary-ML pathologies **one level up**: a *task-distribution shift* becomes a domain-generalization problem over tasks, and a *task stream* causes meta-level forgetting.

**Intentionally deferred extensions.**
- **Meta-learning for Domain Generalization** via an episodic pseudo-target-domain — reuses the MAML episodic machinery from Demo 1.
- The full **higher-order task-distribution-shift** treatment (task augmentation, ICML 2021) and the bi-level *learn to teach* that differentiates the student's test loss through training.

**References.** MAML (Finn et al., 2017); *BERT Learns to Teach* / Meta Knowledge Distillation (ACL 2022); EWC (Kirkpatrick et al., 2017), SI, MAS; *Online Meta-Learning* (Finn et al., ICML 2019); task augmentation (ICML 2021).

---
*All three demos ran CPU-friendly and end-to-end with **Run All** — meta-learning made concrete.*